# Stacking de niveles: que el modelo elija en qué nivel confiar

## La pregunta

Para predecir las toneladas de un producto hay dos caminos, y los dos existen ya:

| | Cómo predice el producto | Dónde está hoy |
|---|---|---|
| **bottom-up** | modela cada par producto-cliente y **suma** | `src/pipe/03_Optuna` |
| **directo** | modela la serie del producto, ya agregada | `nat_exp/07` |

Ninguno domina al otro *a priori*. El bottom-up entrena con 9 millones de filas en vez
de 28 mil, y al sumar predicciones los errores de distinto signo se cancelan. El directo
trabaja con una serie mucho menos ruidosa y no necesita acertarle a cada cliente para
acertarle al total.

**La idea de este notebook es no elegir.** Se calcula la predicción bottom-up, se agrega
por producto, y esa columna entra **como feature** de un modelo a nivel producto. Así el
modelo aprende *cuándo* confiarle a cada nivel — por ejemplo, confiar en el agregado del
nivel fino cuando el producto tiene muchos clientes, y en su propia serie cuando tiene
pocos.

En la literatura es *reconciliación jerárquica por stacking*. Y las tres ramas se miden
acá con la misma partición y la misma métrica:

| Rama | Predicción del producto |
|---|---|
| **1 bottom-up** | suma de las predicciones producto-cliente |
| **2 directo** | modelo de producto, sin la columna del nivel fino |
| **3 stacking** | modelo de producto **con** `pred_pc_sum` como feature |

Más el baseline `ma3` como piso de referencia.

## El problema difícil: el leakage del stacking

Éste es el punto donde este tipo de experimento sale mal casi siempre, así que conviene
entenderlo antes de mirar una línea de código.

Si entreno el modelo producto-cliente con **todos** los meses de train y después uso sus
predicciones como feature del modelo de producto **en esos mismos meses**, esa feature es
*in-sample*: el modelo pc ya vio las respuestas de esas filas, así que su predicción es
artificialmente buena ahí. El modelo de producto va a aprender a confiarle ciegamente, y
en producción — donde la predicción es genuinamente fuera de muestra y peor — se va a
equivocar sistemáticamente.

La solución es **validación cruzada temporal por bloques** (*walk-forward*). Los meses de
train se parten en bloques, y para cada bloque se entrena un modelo pc **sólo con meses
anteriores**, respetando el gap del horizonte. Así `pred_pc_sum` es fuera de muestra en
**todas** las filas donde se la usa para entrenar.

```
bloque 1:  train pc [........]  gap  predice ####
bloque 2:  train pc [.............]  gap  predice ####
bloque 3:  train pc [..................]  gap  predice ####
```

La condición exacta, que el notebook verifica con un assert: **el mes objetivo de
cualquier fila de entrenamiento del modelo pc tiene que ser anterior al primer mes del
bloque que ese modelo predice.**

Y una consecuencia honesta que hay que tener presente: los bloques tempranos se predicen
con menos historia que los tardíos, así que la calidad de `pred_pc_sum` **no es
homogénea** a lo largo del train. Es inherente al walk-forward y no se puede eliminar,
sólo saber que está.

## Hiperparámetros: Optuna en vez de fijarlos a mano

`lgbm_p` (nivel producto, ~28 mil filas) se afina con Optuna directo: es barato, así que
se busca sobre las mismas filas/features que usa la rama 2b (la comparación limpia contra
la rama 3), y el resultado se reusa en las ramas 2/2b/3 por igual -- si cada rama tuviera
sus propios hiperparámetros, una mejora en la rama 3 podría deberse a un mejor ajuste en
vez de al aporte real de `pred_pc_sum`, y se perdería la comparación limpia que es el
punto central del notebook.

`lgbm_pc` (nivel producto-cliente, ~9 millones de filas) es demasiado caro para buscarle
hiperparámetros directo: ya se reentrena 7 veces con parámetros fijos (nota original: 30
min a 2 hs). Meterle Optuna encima repetiría eso una vez por trial. La salida: buscar los
hiperparámetros sobre una **muestra de los clientes de mayor volumen** (`top_clientes_pc`,
default 20) con un solo split train/val (no el walk-forward completo) -- mucho más barato
-- y una vez encontrados, usarlos para el walk-forward real con TODOS los clientes. Es una
aproximación (los mejores hiperparámetros en una muestra chica no tienen por qué ser
exactamente los óptimos en la población completa), pero muchísimo más barata que la
alternativa, y el objetivo de Optuna acá (el mismo WAPE a nivel producto que mide el resto
del notebook, no un MAE genérico a nivel fila) apunta al lugar correcto.


## 0 — Ambiente


In [ ]:
import gc, json, os, shutil, subprocess, time
from pathlib import Path

import numpy as np
import polars as pl
import pandas as pd
import lightgbm as lgb
import optuna
from tqdm.auto import tqdm

optuna.logging.set_verbosity(optuna.logging.WARNING)


def resolver_bucket() -> Path:
    env = os.environ.get("LABO3_BUCKET")
    if env and Path(env).expanduser().exists():
        return Path(env).expanduser().resolve()
    for cand in (Path.home() / "buckets" / "b1",
                 "/content/buckets/b1", "/home/ds/buckets/b1"):
        if Path(cand).is_dir():
            return Path(cand)
    raise RuntimeError("No encontre el bucket. Defini LABO3_BUCKET.")


BUCKET   = resolver_bucket()
DIR_RAW  = BUCKET / "datasets"
RUTA_EXP = BUCKET / "exp_stacking"
RUTA_EXP.mkdir(parents=True, exist_ok=True)
print(f"BUCKET: {BUCKET}\nsalida: {RUTA_EXP}")


## 1 — Palancas


In [ ]:
def rango_meses(desde: int, hasta: int) -> list:
    a, b = (desde // 100) * 12 + desde % 100, (hasta // 100) * 12 + hasta % 100
    return [((m - 1) // 12) * 100 + ((m - 1) % 12) + 1 for m in range(a, b + 1)]


PARAM = {
    'solo_productos_target': True,
    # None = todos los productos. Un numero para probar el notebook rapido: el modelo
    # pc se entrena una vez por bloque, asi que con todos los datos esta celda es la
    # mas cara de todo el repo.
    'muestra_productos': None,

    'horizonte': 2,
    'max_lags': 12,

    'meses_train': rango_meses(201701, 201905),
    'meses_val':   [201907, 201908],
    'meses_test':  [201910],
    'reentrenar_con_val_para_test': True,

    # ── El walk-forward que genera pred_pc_sum fuera de muestra ──────────
    'n_bloques_oof': 4,
    'min_meses_historia': 12,

    # ── Optuna para lgbm_p (nivel producto, barato -- se busca directo) ──
    'n_trials_p': 30,
    'regularizacion_optuna': 'normal',   # 'normal' | 'fuerte'

    # ── Optuna para lgbm_pc (nivel producto-cliente, caro -- se busca en
    # una MUESTRA de los clientes de mayor volumen, un solo split train/val,
    # no el walk-forward completo) ───────────────────────────────────────
    'n_trials_pc': 20,
    'top_clientes_pc': 20,

    'baseline': 'ma3',

    'periodo_objetivo': 202002,
    'clip_min': 0.0,
    'kaggle_competition': 'labo-iii-2026-rosario',
    'submit': False,
    'semilla': 102191,
    'sufijo': '',

    'forzar': set(),   # 'optuna_p' | 'optuna_pc' -- fuerza recalculo aunque haya cache
}

H = PARAM['horizonte']
L = PARAM['max_lags']

EXPERIMENTO = (f"stacking_{L}lags_{PARAM['n_bloques_oof']}bloques"
               f"_base-{PARAM['baseline']}"
               f"_val{PARAM['meses_val'][0]}-{PARAM['meses_val'][-1]}"
               f"_test{PARAM['meses_test'][0]}"
               + (f"_{PARAM['sufijo']}" if PARAM['sufijo'] else ""))
DIR_OUT = RUTA_EXP / EXPERIMENTO
DIR_OUT.mkdir(parents=True, exist_ok=True)
print(f"EXPERIMENTO: {EXPERIMENTO}")


## 2 — Los dos paneles

Se construyen desde la misma fuente con **la misma función de features**, cambiando sólo
las claves. Eso garantiza que la comparación entre niveles sea de *nivel de agregación* y
no de qué features tiene cada uno.


In [ ]:
t0 = time.time()

sell = pl.read_csv(DIR_RAW / "sell-in.txt.gz", separator="\t")
prod = (pl.read_csv(DIR_RAW / "tb_productos.txt", separator="\t")
          .unique(subset=["product_id"]))
target_ids = pl.read_csv(DIR_RAW / "product_id_apredecir201912.txt")["product_id"].to_list()

if PARAM['solo_productos_target']:
    sell = sell.filter(pl.col("product_id").is_in(target_ids))
if PARAM['muestra_productos']:
    _top = (sell.group_by("product_id").agg(pl.col("tn").sum().alias("t"))
                .sort("t", descending=True).head(PARAM['muestra_productos'])["product_id"])
    sell = sell.filter(pl.col("product_id").is_in(_top.to_list()))

print(f"sell-in: {sell.height:,} filas . {sell['product_id'].n_unique()} productos "
      f". {sell['customer_id'].n_unique()} clientes")

CATS = ["cat1", "cat2", "cat3", "brand"]


def construir_panel(keys):
    """Panel densificado dentro de la vida de cada serie, con la clave que se le pase."""
    p = (sell.group_by(keys + ["periodo"])
             .agg(pl.col("tn").sum().alias("tn"),
                  pl.col("cust_request_qty").sum().alias("req_qty"),
                  pl.col("customer_id").n_unique().alias("n_clientes"))
             .with_columns((((pl.col("periodo") // 100) * 12)
                            + (pl.col("periodo") % 100)).alias("m")))
    vida = p.group_by(keys).agg(pl.col("m").min().alias("m_nace"),
                                pl.col("m").max().alias("m_muere"))
    grilla = (vida.with_columns(pl.int_ranges("m_nace", pl.col("m_muere") + 1).alias("m"))
                  .explode("m").select(keys + ["m"]))
    p = (grilla.join(p.drop("periodo"), on=keys + ["m"], how="left")
               .with_columns(pl.col("tn").fill_null(0.0), pl.col("req_qty").fill_null(0),
                             pl.col("n_clientes").fill_null(0))
               .join(vida, on=keys, how="left")
               .join(prod.select("product_id", *CATS, "sku_size"),
                     on="product_id", how="left")
               .with_columns(
                   ((((pl.col("m") - 1) // 12) * 100) + ((pl.col("m") - 1) % 12) + 1)
                     .alias("periodo"),
                   pl.when(pl.col("m") >= pl.col("m_nace"))
                     .then(pl.col("m") - pl.col("m_nace")).otherwise(-1).alias("edad"))
               .sort(keys + ["m"]))
    return p


def agregar_features(p, keys):
    """Lags, promedios moviles, shares e indices. Todo causal: solo mira hasta t."""
    for niv in ("cat1", "cat3"):
        t = p.group_by([niv, "m"]).agg(pl.col("tn").sum().alias(f"tn_{niv}"))
        p = p.join(t, on=[niv, "m"], how="left")
    mer = p.group_by("m").agg(pl.col("tn").sum().alias("tn_mercado"))
    p = p.join(mer, on="m", how="left")

    def div(num, den, alias):
        return (pl.when(pl.col(den).abs() > 1e-9)
                  .then(pl.col(num) / pl.col(den)).otherwise(0.0).alias(alias))

    shares = ["sh_cat1", "sh_cat3", "sh_mercado"]
    p = p.with_columns(div("tn", "tn_cat1", "sh_cat1"),
                       div("tn", "tn_cat3", "sh_cat3"),
                       div("tn", "tn_mercado", "sh_mercado"))

    p = p.sort(keys + ["m"]).with_columns(
        *[pl.col("tn").shift(k).over(keys).alias(f"tn_lag{k}") for k in range(1, L + 1)],
        *[pl.col("tn").rolling_mean(w).over(keys).alias(f"tn_ma{w}") for w in (3, 6, 12)],
        *[pl.col(s).shift(1).over(keys).alias(f"{s}_lag1") for s in shares],
        *[pl.col(s).rolling_mean(3).over(keys).alias(f"{s}_ma3") for s in shares],
        pl.col("n_clientes").rolling_mean(3).over(keys).alias("n_clientes_ma3"),
        pl.col("tn").cum_max().over(keys).alias("tn_pico"),
        (pl.col("tn") > 0).cast(pl.Int8).alias("vendio"),
    )
    p = p.with_columns(
        *[(pl.col(s) - pl.col(f"{s}_ma3")).alias(f"{s}_dma3") for s in shares],
        (pl.col("tn") - pl.col("tn_ma3")).alias("tn_dma3"),
        pl.col("vendio").rolling_mean(6).over(keys).alias("frac_venta_6"),
        (pl.col("periodo") % 100).alias("mes_del_anio"),
        (pl.col("edad").is_between(0, 6)).cast(pl.Int8).alias("es_nuevo"),
        pl.when(pl.col("tn_ma3").abs() > 1e-9)
          .then((pl.col("tn") / pl.col("tn_ma3")).clip(0, 10))
          .otherwise(None).alias("idx_vs_ma3"),
        pl.when(pl.col("tn_pico").abs() > 1e-9)
          .then((pl.col("tn") / pl.col("tn_pico")).clip(0, 10))
          .otherwise(None).alias("idx_vs_pico"),
    )
    p = p.sort(keys + ["m"]).with_columns(
        pl.col("tn").shift(-H).over(keys).alias("clase_tn"),
        ((((pl.col("m") + H - 1) // 12) * 100) + ((pl.col("m") + H - 1) % 12) + 1)
          .alias("periodo_objetivo"))
    p = p.with_columns([pl.col(c).cast(pl.Utf8).fill_null("NA").cast(pl.Categorical)
                        for c in CATS])
    return p


KEYS_PC = ["product_id", "customer_id"]
KEYS_P = ["product_id"]

pc = agregar_features(construir_panel(KEYS_PC), KEYS_PC)
pp = agregar_features(construir_panel(KEYS_P), KEYS_P)

NO_FEAT = {"m", "m_nace", "m_muere", "periodo", "clase_tn", "periodo_objetivo"}
FEAT_PC = [c for c in pc.columns if c not in NO_FEAT | set(KEYS_PC)]
FEAT_P = [c for c in pp.columns if c not in NO_FEAT | set(KEYS_P)]

print(f"\npanel pc: {pc.height:,} filas x {len(FEAT_PC)} features")
print(f"panel p : {pp.height:,} filas x {len(FEAT_P)} features")
print(f"[{time.time()-t0:.0f}s]")


## 3 — Partición y métrica


In [ ]:
def a_m(p):
    return (p // 100) * 12 + (p % 100)


sup_pc = pc.filter(pl.col("clase_tn").is_not_null())
sup_p = pp.filter(pl.col("clase_tn").is_not_null())
periodos_sup = sorted(sup_p["periodo"].unique().to_list())

MESES_TRAIN = [m for m in PARAM['meses_train'] if m in periodos_sup]
MESES_VAL = [m for m in PARAM['meses_val'] if m in periodos_sup]
MESES_TEST = [m for m in PARAM['meses_test'] if m in periodos_sup]
MESES_INFER = sorted(pp.filter(pl.col("clase_tn").is_null())["periodo"].unique().to_list())[-H:]

errores = []


def chk(ok, msg):
    print(f"  [{'ok   ' if ok else 'ERROR'}] {msg}")
    if not ok:
        errores.append(msg)


print("CONTROL DE LEAKAGE (particion)")
print("=" * 74)
for a, b, na, nb in ((MESES_TRAIN, MESES_VAL, "train", "val"),
                     (MESES_VAL, MESES_TEST, "val", "test")):
    g = a_m(min(b)) - a_m(max(a))
    chk(g >= H, f"gap {na}({max(a)}) -> {nb}({min(b)}) = {g} >= horizonte {H}")
chk(max(MESES_TRAIN) < min(MESES_VAL) < max(MESES_VAL) < min(MESES_TEST),
    "orden cronologico train < val < test")
for nombre, panel, keys in (("pc", pc, KEYS_PC), ("p", pp, KEYS_P)):
    _u = (panel.filter(pl.col("clase_tn").is_not_null())
               .group_by(keys).agg(pl.len().alias("n")).sort("n", descending=True).head(1))
    _f = pl.all_horizontal([pl.col(c) == _u[c][0] for c in keys])
    _s = panel.filter(_f).sort("m")
    _tn, _cl = _s["tn"].to_list(), _s["clase_tn"].to_list()
    _mal = [i for i in range(len(_tn) - H)
            if _cl[i] is not None and abs(_cl[i] - _tn[i + H]) > 1e-9]
    chk(not _mal, f"panel {nombre}: clase_tn[i] == tn[i+{H}] ({len(_mal)} discrepancias)")
print("=" * 74)
if errores:
    raise RuntimeError(f"Leakage: {errores}")


def wape(y_real, y_pred, ids=None) -> float:
    yr = np.asarray(y_real, dtype=np.float64)
    yp = np.maximum(np.asarray(y_pred, dtype=np.float64), 0.0)
    if ids is not None:
        _, inv = np.unique(np.asarray(ids), return_inverse=True)
        yr, yp = np.bincount(inv, weights=yr), np.bincount(inv, weights=yp)
    den = np.abs(yr).sum()
    return float("nan") if den == 0 else float(np.abs(yr - yp).sum() / den)


def wape_p(b, pred):
    """WAPE a nivel producto. b es un bloque del panel de PRODUCTO."""
    return wape(b["clase_tn"].to_numpy(), pred, b["product_id"].to_numpy())


BASE_P = dict(objective="regression", metric="mae", verbosity=-1, subsample_freq=1,
              seed=PARAM['semilla'], n_jobs=-1, deterministic=True, force_row_wise=True)


def espacio_hiper(trial, regularizacion='normal'):
    if regularizacion == 'fuerte':
        return {**BASE_P,
            'num_leaves': trial.suggest_int('num_leaves', 8, 64),
            'max_depth': trial.suggest_int('max_depth', 3, 7),
            'learning_rate': trial.suggest_float('learning_rate', 5e-3, 0.1, log=True),
            'n_estimators': trial.suggest_int('n_estimators', 100, 600),
            'min_child_samples': trial.suggest_int('min_child_samples', 30, 200),
            'subsample': trial.suggest_float('subsample', 0.5, 0.9),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.9),
            'reg_alpha': trial.suggest_float('reg_alpha', 0.1, 20.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 20.0, log=True),
        }
    return {**BASE_P,
        'num_leaves': trial.suggest_int('num_leaves', 20, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 100, 800),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
    }


print(f"TRAIN {len(MESES_TRAIN)} meses . VAL {MESES_VAL} . TEST {MESES_TEST} "
      f". INFER {MESES_INFER}")


## 4 — Optuna para `lgbm_p` (nivel producto, barato)

Se busca sobre las filas/features de la comparacion limpia (rama 2b: los meses que
despues van a tener `pred_pc_sum`, features SIN esa columna) -- asi el resultado sirve
igual de bien para las ramas 2, 2b y 3, y la comparacion entre ellas sigue aislando
unicamente el aporte de `pred_pc_sum`. No hace falta esperar al walk-forward: esta
busqueda no usa esa columna.


In [ ]:
_hist = PARAM['min_meses_historia']
_resto = MESES_TRAIN[_hist:]
if len(_resto) < PARAM['n_bloques_oof']:
    raise RuntimeError(
        f"Solo quedan {len(_resto)} meses despues de reservar {_hist} de historia: "
        f"no alcanzan para {PARAM['n_bloques_oof']} bloques. Baja 'min_meses_historia' "
        f"o 'n_bloques_oof'.")
BLOQUES = [list(x) for x in np.array_split(np.array(_resto), PARAM['n_bloques_oof'])]
print(f"{len(MESES_TRAIN)} meses de train: {_hist} de historia reservada + "
      f"{len(_resto)} repartidos en {len(BLOQUES)} bloques")


def bloque_p_base(meses):
    return pp.filter(pl.col("periodo").is_in(meses) & pl.col("clase_tn").is_not_null())


path_hiper_p = DIR_OUT / "hiper_lgbm_p.json"
if path_hiper_p.exists() and 'optuna_p' not in PARAM['forzar']:
    print(f"[lgbm_p] cache encontrada -> se saltea: {path_hiper_p}")
    with open(path_hiper_p) as f:
        hiper_p = json.load(f)
else:
    print("[lgbm_p] corriendo Optuna...")
    _tr_p = bloque_p_base(_resto).to_pandas()
    _va_p = bloque_p_base(MESES_VAL)
    _va_p_pd = _va_p.to_pandas()

    def objective_p(trial):
        params = espacio_hiper(trial, PARAM['regularizacion_optuna'])
        modelo = lgb.LGBMRegressor(**params)
        modelo.fit(_tr_p[FEAT_P], _tr_p["clase_tn"], categorical_feature=CATS)
        pred = modelo.predict(_va_p_pd[FEAT_P])
        return wape_p(_va_p, pred)

    db_local = Path.home() / "optuna_stacking_lgbm_p.db"
    db_bucket = RUTA_EXP / "optuna_lgbm_p.db"
    if db_bucket.exists() and not db_local.exists():
        shutil.copy(db_bucket, db_local)
    study_p = optuna.create_study(direction="minimize",
                                  sampler=optuna.samplers.TPESampler(seed=PARAM['semilla']),
                                  study_name="stacking_lgbm_p", storage=f"sqlite:///{db_local}",
                                  load_if_exists=True)
    print(f"  trials previos: {len(study_p.trials)}   corriendo {PARAM['n_trials_p']} nuevos")
    with tqdm(total=PARAM['n_trials_p'], desc="Optuna lgbm_p") as pbar:
        def cb_p(study, trial):
            pbar.update(1)
            pbar.set_postfix({"mejor wape": f"{study.best_value:.5f}"})
        study_p.optimize(objective_p, n_trials=PARAM['n_trials_p'], callbacks=[cb_p])
    shutil.copy(db_local, db_bucket)

    hiper_p = study_p.best_params
    with open(path_hiper_p, "w", encoding="utf-8") as f:
        json.dump({"hiperparametros": hiper_p, "wape_val": study_p.best_value,
                  "n_trials_total": len(study_p.trials)}, f, indent=2, ensure_ascii=False)
    print(f"  [lgbm_p] mejor wape_val: {study_p.best_value:.5f}")

PARAM['lgbm_p'] = hiper_p
print(f"lgbm_p (Optuna): {PARAM['lgbm_p']}")


## 5 — Optuna para `lgbm_pc` (nivel producto-cliente, muestra top-N clientes)

Un solo split train($\le$train)/val, sobre los `top_clientes_pc` clientes de mayor volumen
-- no el walk-forward completo, que sería repetir el paso más caro del notebook una vez
por trial. El objetivo es el mismo WAPE a nivel producto que mide el resto del notebook
(agregando las predicciones pc de la muestra), no un MAE genérico fila por fila.


In [ ]:
def agregar_a_producto(bloque_pc, pred):
    """Suma las predicciones producto-cliente por (producto, mes). Es el bottom-up."""
    return (bloque_pc.select("product_id", "periodo")
                     .with_columns(pl.Series("pred", pred))
                     .group_by(["product_id", "periodo"])
                     .agg(pl.col("pred").sum().alias("pred_pc_sum")))


top_clientes = (sell.group_by("customer_id").agg(pl.col("tn").sum().alias("t"))
                    .sort("t", descending=True).head(PARAM['top_clientes_pc'])
                    ["customer_id"].to_list())
sup_pc_muestra = sup_pc.filter(pl.col("customer_id").is_in(top_clientes))
print(f"muestra Optuna lgbm_pc: {len(top_clientes)} clientes, {sup_pc_muestra.height:,} filas "
     f"(de {sup_pc.height:,} totales)")

path_hiper_pc = DIR_OUT / "hiper_lgbm_pc.json"
if path_hiper_pc.exists() and 'optuna_pc' not in PARAM['forzar']:
    print(f"[lgbm_pc] cache encontrada -> se saltea: {path_hiper_pc}")
    with open(path_hiper_pc) as f:
        hiper_pc = json.load(f)
else:
    print("[lgbm_pc] corriendo Optuna (sobre la muestra)...")
    _tr_pc = sup_pc_muestra.filter(pl.col("periodo").is_in(MESES_TRAIN)).to_pandas()
    _va_pc = sup_pc_muestra.filter(pl.col("periodo").is_in(MESES_VAL))
    _va_pc_pd = _va_pc.to_pandas()
    _real_val_p = pp.filter(pl.col("periodo").is_in(MESES_VAL) & pl.col("clase_tn").is_not_null())

    def objective_pc(trial):
        params = espacio_hiper(trial, PARAM['regularizacion_optuna'])
        modelo = lgb.LGBMRegressor(**params)
        modelo.fit(_tr_pc[FEAT_PC], _tr_pc["clase_tn"], categorical_feature=CATS)
        pred = np.maximum(modelo.predict(_va_pc_pd[FEAT_PC]), 0.0)
        agregado = agregar_a_producto(_va_pc, pred)
        comparado = _real_val_p.join(agregado, on=["product_id", "periodo"], how="inner")
        if comparado.height == 0:
            raise optuna.TrialPruned()
        return wape(comparado["clase_tn"].to_numpy(), comparado["pred_pc_sum"].fill_null(0.0).to_numpy(),
                   comparado["product_id"].to_numpy())

    db_local = Path.home() / "optuna_stacking_lgbm_pc.db"
    db_bucket = RUTA_EXP / "optuna_lgbm_pc.db"
    if db_bucket.exists() and not db_local.exists():
        shutil.copy(db_bucket, db_local)
    study_pc = optuna.create_study(direction="minimize",
                                   sampler=optuna.samplers.TPESampler(seed=PARAM['semilla']),
                                   study_name="stacking_lgbm_pc", storage=f"sqlite:///{db_local}",
                                   load_if_exists=True)
    print(f"  trials previos: {len(study_pc.trials)}   corriendo {PARAM['n_trials_pc']} nuevos")
    with tqdm(total=PARAM['n_trials_pc'], desc="Optuna lgbm_pc (muestra)") as pbar:
        def cb_pc(study, trial):
            pbar.update(1)
            try:
                pbar.set_postfix({"mejor wape": f"{study.best_value:.5f}"})
            except ValueError:
                pass
        study_pc.optimize(objective_pc, n_trials=PARAM['n_trials_pc'], callbacks=[cb_pc])
    shutil.copy(db_local, db_bucket)

    hiper_pc = study_pc.best_params
    with open(path_hiper_pc, "w", encoding="utf-8") as f:
        json.dump({"hiperparametros": hiper_pc, "wape_val_muestra": study_pc.best_value,
                  "n_trials_total": len(study_pc.trials), "top_clientes_pc": PARAM['top_clientes_pc']},
                 f, indent=2, ensure_ascii=False)
    print(f"  [lgbm_pc] mejor wape_val (muestra de {PARAM['top_clientes_pc']} clientes): {study_pc.best_value:.5f}")

PARAM['lgbm_pc'] = hiper_pc
print(f"lgbm_pc (Optuna, muestra): {PARAM['lgbm_pc']}")


## 6 — `pred_pc_sum` fuera de muestra, por walk-forward (con `lgbm_pc` ya afinado)

Ésta es la celda que hace el trabajo difícil, y **la más cara de todo el repo**: entrena
un modelo producto-cliente por bloque, ahora con los hiperparámetros que encontró Optuna
en la sección anterior (sobre la muestra) en vez de un valor fijo a mano.

Para cada bloque de meses se entrena con **todo lo anterior**, respetando el gap del
horizonte, y se predice el bloque. La condición que se verifica con un assert en cada
vuelta: **el mes objetivo más alto de las filas de entrenamiento tiene que ser anterior
al primer mes del bloque predicho.** Si eso no se cumple, la feature está contaminada.


In [ ]:
def fit_pc(meses, params):
    b = sup_pc.filter(pl.col("periodo").is_in(meses)).to_pandas()
    m = lgb.LGBMRegressor(**{**BASE_P, **params})
    m.fit(b[FEAT_PC], b["clase_tn"], categorical_feature=CATS)
    return m


def predecir_pc(modelo, bloque_pc):
    return np.maximum(modelo.predict(bloque_pc.to_pandas()[FEAT_PC]), 0.0)


t0 = time.time()
oof = []
for i, bl in enumerate(BLOQUES, 1):
    m0 = a_m(bl[0])
    meses_fit = [m for m in MESES_TRAIN if a_m(m) + H < m0]
    if not meses_fit:
        raise RuntimeError(f"bloque {i} ({bl[0]}) sin meses de entrenamiento validos")
    _obj_max = max(a_m(m) + H for m in meses_fit)
    assert _obj_max < m0, (f"bloque {i}: el objetivo mas alto de train es "
                           f"{_obj_max} y el bloque arranca en {m0}")
    mod = fit_pc(meses_fit, PARAM['lgbm_pc'])
    blq = sup_pc.filter(pl.col("periodo").is_in(bl))
    oof.append(agregar_a_producto(blq, predecir_pc(mod, blq)))
    print(f"  bloque {i}/{len(BLOQUES)}: fit con {len(meses_fit):2d} meses "
          f"(hasta {max(meses_fit)}) -> predice {bl[0]}..{bl[-1]}   "
          f"[{time.time()-t0:.0f}s]", flush=True)
    del mod, blq
    gc.collect()

OOF = pl.concat(oof)
print(f"\npred_pc_sum fuera de muestra: {OOF.height:,} filas producto-mes")
print(f"cubre los meses {sorted(OOF['periodo'].unique().to_list())[:3]}..."
      f"{sorted(OOF['periodo'].unique().to_list())[-2:]}")


def pc_sum_para(meses_eval, meses_fit):
    mod = fit_pc(meses_fit, PARAM['lgbm_pc'])
    blq = (sup_pc if meses_eval[0] in periodos_sup else pc).filter(
        pl.col("periodo").is_in(meses_eval))
    r = agregar_a_producto(blq, predecir_pc(mod, blq))
    del mod, blq
    gc.collect()
    return r


MESES_FIT_TEST = (MESES_TRAIN + MESES_VAL) if PARAM['reentrenar_con_val_para_test'] else MESES_TRAIN
t0 = time.time()
PC_VAL = pc_sum_para(MESES_VAL, MESES_TRAIN)
print(f"pred_pc_sum para val  [{time.time()-t0:.0f}s]")
PC_TEST = pc_sum_para(MESES_TEST, MESES_FIT_TEST)
print(f"pred_pc_sum para test [{time.time()-t0:.0f}s]")
PC_INFER = pc_sum_para(MESES_INFER, sorted(periodos_sup))
print(f"pred_pc_sum para inferencia [{time.time()-t0:.0f}s]")


## 7 — Las tres ramas (con `lgbm_p` ya afinado)

Se pega `pred_pc_sum` al panel de producto y se entrenan las tres variantes, con los
hiperparámetros de `lgbm_p` que encontró Optuna en la sección 4. La rama 3 usa exactamente
las mismas features que la 2 **más una columna**, así que la diferencia entre ellas aísla
el aporte del nivel fino.


In [ ]:
PC_TODO = pl.concat([OOF, PC_VAL, PC_TEST])
pp_s = pp.join(PC_TODO, on=["product_id", "periodo"], how="left")

MESES_STACK = sorted(OOF["periodo"].unique().to_list())
print(f"meses con pred_pc_sum fuera de muestra: {len(MESES_STACK)} "
      f"({MESES_STACK[0]}..{MESES_STACK[-1]})")
print(f"meses de train excluidos de la rama 3: "
      f"{[m for m in MESES_TRAIN if m not in MESES_STACK]}")

FEAT_STACK = FEAT_P + ["pred_pc_sum"]


def bloque_p(meses, con_stack=False):
    b = pp_s.filter(pl.col("periodo").is_in(meses) & pl.col("clase_tn").is_not_null())
    if con_stack:
        b = b.filter(pl.col("pred_pc_sum").is_not_null())
    return b


def fit_p(meses, feats, params, con_stack=False, semilla=None):
    b = bloque_p(meses, con_stack).to_pandas()
    p = {**BASE_P, **params}
    if semilla is not None:
        p["seed"] = semilla
    m = lgb.LGBMRegressor(**p)
    m.fit(b[feats], b["clase_tn"], categorical_feature=CATS)
    return m


def baseline_de(b):
    return np.maximum(b[f"tn_{PARAM['baseline']}"].fill_null(0.0).to_numpy(), 0.0)


def tres_ramas(meses_fit, b_eval):
    """Devuelve {rama: pred_tn} sobre el bloque de evaluacion (nivel producto)."""
    ev = b_eval.to_pandas()
    out, mods = {}, {}

    out["0_baseline"] = baseline_de(b_eval)
    out["1_bottom_up"] = b_eval["pred_pc_sum"].fill_null(0.0).to_numpy()

    m2 = fit_p(meses_fit, FEAT_P, PARAM['lgbm_p'])
    out["2_directo"] = m2.predict(ev[FEAT_P]); mods["2_directo"] = m2

    meses3 = [m for m in meses_fit if m in MESES_STACK]
    m3 = fit_p(meses3, FEAT_STACK, PARAM['lgbm_p'], con_stack=True)
    out["3_stacking"] = m3.predict(ev[FEAT_STACK]); mods["3_stacking"] = m3

    m2b = fit_p(meses3, FEAT_P, PARAM['lgbm_p'], con_stack=True)
    out["2b_directo_mismos_meses"] = m2b.predict(ev[FEAT_P])
    mods["2b_directo_mismos_meses"] = m2b

    return out, mods


t0 = time.time()
pred_val, mod_val = tres_ramas(MESES_TRAIN, bloque_p(MESES_VAL))
va_p = bloque_p(MESES_VAL)
print(f"[{time.time()-t0:.0f}s]\n")

RAMAS = list(pred_val)
print(f"{'rama':14s} {'WAPE val':>10s}")
print("-" * 26)
wape_val = {r: wape_p(va_p, pred_val[r]) for r in RAMAS}
for r in RAMAS:
    print(f"{r:14s} {wape_val[r]:10.5f}")


## 8 — Resultados en test, y dónde gana cada nivel

El corte por **cantidad de clientes del producto** es la pregunta de fondo: la hipótesis
es que el nivel fino ayuda donde hay muchos clientes — porque ahí la suma de muchas
predicciones cancela ruido — y estorba donde hay pocos, porque la serie del par es casi
la del producto pero más ruidosa.


In [ ]:
te_p = bloque_p(MESES_TEST)
pred_test, mod_test = tres_ramas(MESES_FIT_TEST, te_p)

print(f"{'rama':14s} {'WAPE val':>10s} {'WAPE test':>10s}")
print("-" * 38)
METRICAS = {}
for r in RAMAS:
    wt = wape_p(te_p, pred_test[r])
    METRICAS[r] = {"val": wape_val[r], "test": wt}
    print(f"{r:14s} {wape_val[r]:10.5f} {wt:10.5f}")

GANADOR = min(METRICAS, key=lambda r: METRICAS[r]['test'])
_b = METRICAS['0_baseline']['test']
print(f"\nganador en test: {GANADOR}")
for r in RAMAS[1:]:
    print(f"  {r:14s} vs baseline: {100*(_b-METRICAS[r]['test'])/_b:+.1f}%")
_d = METRICAS['2b_directo_mismos_meses']['test']
_s = METRICAS['3_stacking']['test']
_d2 = METRICAS['2_directo']['test']
print(f"\nAPORTE LIMPIO del nivel fino (3_stacking vs 2b, mismos meses): "
      f"{100*(_d-_s)/_d:+.1f}%")
print("Positivo = la columna del nivel fino agrega informacion que el nivel producto")
print("no tenia. Cerca de 0 = los dos niveles dicen lo mismo.")
print(f"\nCosto de perder meses de train por el walk-forward: "
      f"2_directo ({len(MESES_TRAIN)} meses) vs 2b ({len([m for m in MESES_TRAIN if m in MESES_STACK])} meses) "
      f"= {100*(_d2-_d)/_d2:+.1f}%")
print("Si ese numero es grande, el walk-forward sale caro en datos: subi 'n_bloques_oof'")
print("o baja 'min_meses_historia'.")

pl.DataFrame([{"rama": r, **METRICAS[r]} for r in RAMAS]).write_csv(DIR_OUT / "ramas.csv")

imp = (pl.DataFrame({"feature": mod_test["3_stacking"].feature_name_,
                     "gain": mod_test["3_stacking"].booster_.feature_importance("gain")})
         .with_columns((100 * pl.col("gain") / pl.col("gain").sum()).round(2).alias("gain_pct"))
         .sort("gain", descending=True))
imp.write_csv(DIR_OUT / "importancia_stacking.csv")
_pos = imp.with_row_index("puesto").filter(pl.col("feature") == "pred_pc_sum")
print(f"\npred_pc_sum en la importancia: puesto {int(_pos['puesto'][0])+1} de {imp.height}, "
      f"{float(_pos['gain_pct'][0]):.1f}% del gain")
print("Si esta al fondo, el modelo la ignora y el stacking no puede aportar nada.")
print(imp.head(10))

n_cli = (sell.group_by("product_id").agg(pl.col("customer_id").n_unique().alias("n_cli")))
det = (te_p.select("product_id", "clase_tn")
           .with_columns(*[pl.Series(r, pred_test[r]) for r in RAMAS])
           .join(n_cli, on="product_id", how="left"))
_q = det.select("product_id", "n_cli").unique()
det = det.join(
    _q.with_columns(_q["n_cli"].qcut(3, labels=["pocos", "medio", "muchos"],
                                     allow_duplicates=True).alias("grupo_clientes")),
    on=["product_id", "n_cli"], how="left")

filas = []
for g in ("pocos", "medio", "muchos"):
    b = det.filter(pl.col("grupo_clientes") == g)
    if b.height < 3:
        continue
    f = {"grupo_clientes": g, "n_productos": b["product_id"].n_unique(),
         "clientes_mediana": int(b["n_cli"].median()),
         "tn_real": round(float(b["clase_tn"].sum()), 1)}
    for r in RAMAS:
        f[r] = round(wape(b["clase_tn"], b[r], b["product_id"]), 4)
    filas.append(f)
por_cli = pl.DataFrame(filas)
print()
print(por_cli)
por_cli.write_csv(DIR_OUT / "por_cantidad_de_clientes.csv")
print("\nLa hipotesis: el nivel fino gana donde hay MUCHOS clientes (la suma de muchas")
print("predicciones cancela ruido) y pierde donde hay pocos.")


## 9 — Entrenamiento final, entrega y registro

Se reentrena la rama ganadora con todos los meses supervisados y se predice el mes
objetivo. Para la rama 3, `pred_pc_sum` de las filas de inferencia sale de un modelo
producto-cliente entrenado con **toda** la historia — que es lo correcto acá: en
producción no hay nada que reservar.


In [ ]:
infer_p = (pp.filter(pl.col("periodo").is_in(MESES_INFER))
             .join(PC_INFER, on=["product_id", "periodo"], how="left"))
print(f"filas de inferencia: {infer_p.height:,}   "
      f"con pred_pc_sum: {int(infer_p['pred_pc_sum'].is_not_null().sum()):,}")

MESES_TODOS = sorted(periodos_sup)
ip = infer_p.to_pandas()

if GANADOR == "0_baseline":
    pred_final = baseline_de(infer_p)
elif GANADOR == "1_bottom_up":
    pred_final = infer_p["pred_pc_sum"].fill_null(0.0).to_numpy()
elif GANADOR == "2_directo":
    pred_final = fit_p(MESES_TODOS, FEAT_P, PARAM['lgbm_p']).predict(ip[FEAT_P])
else:
    meses3 = [m for m in MESES_TODOS if m in MESES_STACK] + MESES_VAL + MESES_TEST
    meses3 = sorted(set(m for m in meses3 if m in periodos_sup))
    pred_final = fit_p(meses3, FEAT_STACK, PARAM['lgbm_p'],
                       con_stack=True).predict(ip[FEAT_STACK])

pred_final = np.maximum(pred_final, PARAM['clip_min'])
pred_infer = infer_p.select("product_id", "periodo", "periodo_objetivo").with_columns(
    pl.Series("tn_pred", pred_final),
    pl.Series("pred_pc_sum", infer_p["pred_pc_sum"].fill_null(0.0).to_numpy()),
    pl.Series("baseline", baseline_de(infer_p)))
pred_infer.write_parquet(DIR_OUT / "predicciones_inferencia.parquet")

print(f"\nrama entregada: {GANADOR}")
print(f"correlacion de la prediccion final con:")
for c in ("pred_pc_sum", "baseline"):
    v = pred_infer[c].to_numpy()
    if v.std() > 0:
        print(f"   {c:14s} {np.corrcoef(pred_final, v)[0,1]:+.4f}")
print("Si la correlacion con el baseline es ~1, el modelo lo esta repitiendo.")

OBJ = PARAM['periodo_objetivo']
obj = pred_infer.filter(pl.col("periodo_objetivo") == OBJ)
if obj.is_empty():
    raise RuntimeError(f"No hay predicciones para {OBJ}. Disponibles: "
                       f"{sorted(pred_infer['periodo_objetivo'].unique().to_list())}")
por_prod = obj.group_by("product_id").agg(pl.col("tn_pred").sum().alias("tn"))
oficiales = pl.read_csv(DIR_RAW / "product_id_apredecir201912.txt")
submit = oficiales.select("product_id").join(por_prod, on="product_id", how="left")
sin_pred = int(submit["tn"].null_count())
submit = submit.with_columns(pl.col("tn").fill_null(0.0)).sort("product_id")

print(f"\nSubmit: {submit.height} filas . sin prediccion (van en 0): {sin_pred}")
if sin_pred > oficiales.height * 0.05:
    print("   ATENCION: mas del 5% de la lista.")
print(f"tn   min {submit['tn'].min():.3f}   media {submit['tn'].mean():.3f}   "
      f"max {submit['tn'].max():.3f}   suma {submit['tn'].sum():,.1f}")

path_submit = DIR_OUT / f"submission_{OBJ}.csv"
submit.write_csv(path_submit)
shutil.copy(path_submit, RUTA_EXP / "submission_ultima.csv")
print(f"Guardado: {path_submit}")


def kaggle_cli(args):
    try:
        r = subprocess.run(["kaggle"] + args, capture_output=True, text=True)
        return r.returncode == 0, (r.stdout or "") + (r.stderr or "")
    except FileNotFoundError:
        return False, "La CLI de kaggle no esta instalada.  pip install kaggle"
    except Exception as e:
        return False, f"{type(e).__name__}: {e}"


if not PARAM['submit']:
    print("\nPARAM['submit'] = False -> no se sube. El CSV ya esta generado.")
else:
    kd = Path.home() / ".kaggle" / "kaggle.json"
    kd.parent.mkdir(parents=True, exist_ok=True)
    if not kd.exists():
        for cand in (BUCKET / "kaggle.json", BUCKET / "kaggle" / "kaggle.json"):
            if cand.exists():
                shutil.copy(cand, kd); kd.chmod(0o600); break
    if not kd.exists():
        print("\nSin credenciales de Kaggle. El CSV ya esta generado.")
    else:
        kd.chmod(0o600)
        msg = f"{GANADOR} | wape_test={METRICAS[GANADOR]['test']:.5f} | Optuna lgbm_p+lgbm_pc"
        ok, salida = kaggle_cli(["competitions", "submit",
                                 "-c", PARAM['kaggle_competition'],
                                 "-f", str(path_submit), "-m", msg])
        print(f"\n{msg}\n{salida}")
        print("Submit enviado." if ok else "NO se pudo subir; el CSV esta en disco.")

resultado = {
    'experimento': EXPERIMENTO,
    'idea': 'stacking de niveles: la prediccion producto-cliente agregada como feature '
            'del modelo de producto, generada fuera de muestra por walk-forward. '
            'lgbm_p afinado con Optuna directo; lgbm_pc afinado con Optuna sobre una '
            'muestra de los top_clientes_pc clientes de mayor volumen.',
    'n_bloques_oof': PARAM['n_bloques_oof'],
    'bloques': [[int(x) for x in b] for b in BLOQUES],
    'meses_con_oof': MESES_STACK,
    'baseline': PARAM['baseline'],
    'lgbm_p_optuna': PARAM['lgbm_p'], 'lgbm_pc_optuna': PARAM['lgbm_pc'],
    'top_clientes_pc': PARAM['top_clientes_pc'],
    'metricas_por_rama': METRICAS, 'rama_ganadora': GANADOR,
    'aporte_nivel_fino_pct': round(100 * (_d - _s) / _d, 3),
    'costo_menos_meses_pct': round(100 * (_d2 - _d) / _d2, 3),
    'pred_pc_sum_puesto': int(_pos['puesto'][0]) + 1,
    'pred_pc_sum_gain_pct': float(_pos['gain_pct'][0]),
    'horizonte': H, 'max_lags': L,
    'meses_train': MESES_TRAIN, 'meses_val': MESES_VAL, 'meses_test': MESES_TEST,
    'meses_inferencia': MESES_INFER, 'periodo_objetivo': OBJ,
    'n_features_p': len(FEAT_P), 'n_features_pc': len(FEAT_PC),
    'filas_pc': int(pc.height), 'filas_p': int(pp.height),
    'n_sin_prediccion': sin_pred, 'tn_total': float(submit['tn'].sum()),
    'semilla': PARAM['semilla'],
}
with open(DIR_OUT / "resultado.json", "w", encoding="utf-8") as f:
    json.dump(resultado, f, indent=2, ensure_ascii=False, default=str)

fila = {'experimento': EXPERIMENTO, 'n_bloques': PARAM['n_bloques_oof'],
        'baseline': PARAM['baseline'], 'ganador': GANADOR,
        **{f"test_{r}": round(METRICAS[r]['test'], 5) for r in RAMAS},
        **{f"val_{r}": round(METRICAS[r]['val'], 5) for r in RAMAS},
        'aporte_nivel_fino_pct': round(100 * (_d - _s) / _d, 2),
        'costo_menos_meses_pct': round(100 * (_d2 - _d) / _d2, 2),
        'pred_pc_sum_gain_pct': float(_pos['gain_pct'][0]),
        'sin_prediccion': sin_pred, 'tn_total': round(float(submit['tn'].sum()), 1)}
path_lb = RUTA_EXP / "leaderboard_stacking.csv"
nueva = pl.DataFrame([fila])
if path_lb.exists():
    viejo = pl.read_csv(path_lb).filter(pl.col("experimento") != EXPERIMENTO)
    nueva = pl.concat([viejo, nueva], how="diagonal_relaxed")
nueva.sort("test_3_stacking").write_csv(path_lb)

print(f"\nArchivos en {DIR_OUT.relative_to(BUCKET)}:")
for p in sorted(DIR_OUT.iterdir()):
    print(f"  - {p.name}")
print(f"\nleaderboard_stacking.csv ({nueva.height}):")
print(nueva.select("baseline", "ganador", "test_1_bottom_up", "test_2_directo",
                   "test_3_stacking", "aporte_nivel_fino_pct"))


## Cómo leer el resultado

La comparación que importa es **rama 2b contra rama 3**: entrenadas con las mismas
filas, las mismas features salvo `pred_pc_sum`, y los MISMOS hiperparámetros (los que
encontró Optuna en la sección 4). Ése es el aporte limpio del nivel fino.

La rama 2 (con todos los meses de train) sirve para otra cosa: comparada con 2b mide
**cuánto cuesta** el walk-forward en meses de entrenamiento perdidos.

| Si… | Significa |
|---|---|
| **3 le gana a 2b** | El nivel producto-cliente tiene información que el nivel producto no. El stacking sirve, y `pred_pc_sum` debería aparecer arriba en la importancia. |
| **3 ≈ 2b** | Los dos niveles dicen lo mismo. Modelar el nivel fino no compra nada, y conviene quedarse con el directo, que es 300 veces más barato. |
| **3 pierde contra 2b** | La columna agrega ruido, o quedó contaminada. Revisá que el assert del walk-forward esté pasando y que `pred_pc_sum_gain_pct` no sea absurdamente alto. |
| **1 le gana a las dos** | El bottom-up puro es mejor que cualquier modelo de producto. Sería el resultado más sorprendente, y diría que la agregación por sí sola cancela más error de lo que cualquier modelo puede aprender. |

### Dos números que hay que mirar antes de creer en el resultado

**`pred_pc_sum_gain_pct`.** Si es 60 % o más, sospechá: significa que el modelo se apoya
casi enteramente en esa columna, que es lo que pasaría si estuviera contaminada. En un
stacking sano queda entre 5 % y 25 %.

**La correlación de la predicción final con el baseline.** Si es 0,99, el modelo está
repitiendo el promedio móvil con pasos intermedios, y la mejora de WAPE es cosmética.

### Sobre los hiperparámetros de `lgbm_pc`

Se buscaron sobre una muestra de `top_clientes_pc` clientes (los de mayor volumen), no
sobre la población completa -- es una aproximación para que la búsqueda sea barata. Si
sospechás que no generalizan bien (por ejemplo, si el aporte del nivel fino sale muy
distinto entre el grupo "pocos clientes" y "muchos clientes" en la sección 8), subir
`top_clientes_pc` es la primera palanca para probar, a costa de una búsqueda más cara.

### El costo, para que no te sorprenda

La sección 6 entrena **`n_bloques_oof` + 3** modelos a nivel producto-cliente. Con todos
los productos son ~9 millones de filas por fit. Con la configuración por defecto son 7
modelos: contá entre 30 minutos y 2 horas según la máquina -- eso no cambió. Lo nuevo es
la sección 5 (Optuna para `lgbm_pc`), que corre `n_trials_pc` fits chicos sobre la
muestra antes de llegar ahí -- mucho más rápido que meterle Optuna al walk-forward
completo, pero igual conviene arrancar con `n_trials_pc` bajo (10-20) la primera vez.

Para probar el notebook primero, `PARAM['muestra_productos'] = 60` lo baja a un par de
minutos y verifica que todo el circuito cierra — pero los WAPE de esa corrida no valen
como resultado.
